# 6주차. WebUI와 Sub-Agent 통합 데모

## 0. 목표

실제 OpenAI API를 호출하는 나나/카나 통합 흐름을 노트북과 Gradio UI에서 실행한다.


## 1. 준비


In [ ]:
import json
import os
from typing import Any

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

load_dotenv()

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env 파일에 OPENAI_API_KEY를 설정한 뒤 다시 실행하세요.")


def make_model(max_tokens: int = 500) -> ChatOpenAI:
    return ChatOpenAI(model=OPENAI_MODEL, temperature=0, max_completion_tokens=max_tokens)


def show_json(value: Any) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


def final_text(agent_result: dict[str, Any]) -> str:
    return agent_result["messages"][-1].content


def extract_tool_trace(agent_result: dict[str, Any]) -> list[dict[str, Any]]:
    trace = []
    for message in agent_result.get("messages", []):
        for call in getattr(message, "tool_calls", []) or []:
            trace.append({"event": "tool_call", "tool_name": call.get("name"), "arguments": call.get("args", {})})
        if getattr(message, "type", None) == "tool":
            trace.append({"event": "tool_result", "tool_name": getattr(message, "name", None), "content": message.content})
    return trace



## 2. 개념

최종 데모는 입력, supervisor가 호출한 sub-agent tool, sub-agent 내부 tool trace, final answer를 한 번에 확인하는 작은 사용자 경험이다.


## 3. 기본 개념 실습

가장 작은 흐름은 나나/카나 sub-agent와 supervisor를 한 노트북에서 함께 준비하는 것이다.


In [ ]:
@tool("personal_create_schedule", description="개인 일정을 생성한다.")
def personal_create_schedule(title: str, date: str, start_time: str) -> str:
    """Create a personal schedule."""
    return json.dumps({"ok": True, "schedule": {"title": title, "date": date, "start_time": start_time}}, ensure_ascii=False)

@tool("memory_save", description="사용자 메모를 저장한다.")
def memory_save(title: str, content: str) -> str:
    """Save a user memory."""
    return json.dumps({"ok": True, "memory": {"title": title, "content": content}}, ensure_ascii=False)

@tool("group_confirm_slot", description="그룹 일정 시간을 확정한다.")
def group_confirm_slot(topic: str, selected_slot: str, members: list[str], reason: str) -> str:
    """Confirm a group schedule slot."""
    return json.dumps({"ok": True, "topic": topic, "selected_slot": selected_slot, "members": members, "reason": reason}, ensure_ascii=False)
nana_agent = create_agent(
    model=make_model(700),
    tools=[personal_create_schedule, memory_save],
    system_prompt="너는 나나다. 오늘은 2026-04-23이다. 상대 날짜는 이 날짜 기준으로 YYYY-MM-DD로 바꾼다. 개인 일정이나 메모 요청에 필요한 도구를 호출한다.",
)

kana_agent = create_agent(
    model=make_model(800),
    tools=[group_confirm_slot],
    system_prompt="너는 카나다. 그룹 응답에서 공통 가능 시간을 찾으면 group_confirm_slot 도구를 호출한다.",
)

@tool("nana_agent", description="개인 일정이나 메모 요청을 나나 sub-agent에게 위임한다.")
def delegate_to_nana(request: str) -> str:
    """Delegate a personal request to Nana sub-agent."""
    agent_result = nana_agent.invoke({"messages": [{"role": "user", "content": request}]})
    return json.dumps({"agent": "nana", "answer": final_text(agent_result), "trace": extract_tool_trace(agent_result)}, ensure_ascii=False)

@tool("kana_agent", description="그룹 일정 조율 요청과 멤버 응답을 카나 sub-agent에게 위임한다.")
def delegate_to_kana(request: str, member_replies: str) -> str:
    """Delegate a group coordination request to Kana sub-agent."""
    message = f"요청: {request}\n그룹 응답:\n{member_replies}"
    agent_result = kana_agent.invoke({"messages": [{"role": "user", "content": message}]})
    return json.dumps({"agent": "kana", "answer": final_text(agent_result), "trace": extract_tool_trace(agent_result)}, ensure_ascii=False)

def make_supervisor(mode: str = "auto"):
    if mode == "personal":
        tools = [delegate_to_nana]
        prompt = "너는 카나메이트 supervisor다. 사용자가 personal 모드를 선택했으므로 nana_agent tool을 호출한다."
    elif mode == "group":
        tools = [delegate_to_kana]
        prompt = "너는 카나메이트 supervisor다. 사용자가 group 모드를 선택했으므로 kana_agent tool을 호출한다."
    else:
        tools = [delegate_to_nana, delegate_to_kana]
        prompt = "너는 카나메이트 supervisor다. 개인 일정/메모 요청은 nana_agent tool을 호출하고, 그룹 일정 조율 요청은 kana_agent tool을 호출한다. 직접 처리하지 말고 반드시 적절한 sub-agent tool을 호출한다."
    return create_agent(model=make_model(1000), tools=tools, system_prompt=prompt)


def delegated_agent_from_trace(agent_result: dict[str, Any]) -> str:
    for event in extract_tool_trace(agent_result):
        if event.get("event") == "tool_call" and event.get("tool_name") in {"nana_agent", "kana_agent"}:
            return event["tool_name"].replace("_agent", "")
    return "unknown"


def delegated_payload_from_trace(agent_result: dict[str, Any]) -> dict[str, Any]:
    for event in extract_tool_trace(agent_result):
        if event.get("event") == "tool_result" and event.get("tool_name") in {"nana_agent", "kana_agent"}:
            return json.loads(event["content"])
    return {}


## 4. 카나메이트 확장 예제

준비한 supervisor를 `run_live_flow`로 감싼다. 이 함수는 WebUI와 최종 Golden Scenario가 함께 사용할 선택 agent, 최종 답변, trace, delegate payload를 반환한다.


In [ ]:
def run_live_flow(student_request: str, member_replies: str = "", mode: str = "auto") -> dict[str, Any]:
    supervisor = make_supervisor(mode)
    content = f"요청: {student_request}\n그룹 응답:\n{member_replies}" if mode in {"auto", "group"} else student_request
    supervisor_result = supervisor.invoke({"messages": [{"role": "user", "content": content}]})
    return {"selected_agent": delegated_agent_from_trace(supervisor_result), "answer": final_text(supervisor_result), "trace": extract_tool_trace(supervisor_result), "delegate_payload": delegated_payload_from_trace(supervisor_result)}

student_request = "팀 멤버들과 발표 리허설 시간을 조율해줘"
member_replies = "민수: 2026-04-24 15:00 가능\n지아: 2026-04-24 15:00 가능"
run_result = run_live_flow(student_request, member_replies, mode="auto")
show_json(run_result)


## 5. 확장 예제 실행

`run_live_flow`를 새 입력으로 실행한다. 학생은 `mini_request`와 `mode`를 바꿔 WebUI에서 확인할 시나리오를 먼저 노트북에서 점검한다.


In [ ]:
mini_request = "프로젝트 발표 장소는 3층 세미나실이라고 메모해줘"
mini_result = run_live_flow(mini_request, mode="personal")
mini_inner_tools = [
    event["tool_name"]
    for event in mini_result["delegate_payload"].get("trace", [])
    if event.get("event") == "tool_call"
]

show_json(mini_result)

assert mini_result["selected_agent"] == "nana"
assert mini_result["answer"]
assert any(event["event"] == "tool_call" and event["tool_name"] == "nana_agent" for event in mini_result["trace"])
assert "memory_save" in mini_inner_tools
print("6주차 확장 예제 실행 통과")


## 6. 코드 작성형 실습(`week06.py`)

이번 실습은 나나/카나 통합 Golden Scenario suite를 같은 주차 과제 파일 `week06.py`의 `run_practice_suite`에서 구현하고 실행한다.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "week06.py").exists():
            return candidate
    raise RuntimeError("repo root를 찾지 못했습니다. 노트북을 repo 안에서 실행하세요.")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from week06 import extract_tool_trace, final_text, show_json, practice_cases, run_practice_suite


### 실행 예시

In [ ]:
practice_suite_results = run_practice_suite(practice_cases)
show_json([
    {
        "name": result["name"],
        "expected_agent": result["expected_agent"],
        "selected_agent": result["selected_agent"],
        "expected_inner_tool": result["expected_inner_tool"],
        "inner_tool_names": result["inner_tool_names"],
        "passed": result["passed"],
    }
    for result in practice_suite_results
])

## 6-0. 실습 자동 점검

아래 셀은 모델 답변 문구가 아니라 trace, structured response, payload처럼 구조화된 값을 확인한다.

In [ ]:
assert len(practice_suite_results) == 3
for result in practice_suite_results:
    assert result["selected_agent"] == result["expected_agent"]
    assert result["expected_inner_tool"] in result["inner_tool_names"]
    assert result["passed"] is True
print("6주차 코드 작성형 실습 통과")

## 6-1. 로컬 Gradio UI 실습

이번 주 Gradio UI도 `week06.py` 안에 들어 있다. 터미널에서 아래 명령을 실행하면 해당 주차 UI가 바로 열린다.

```bash
python week06.py
```

노트북 안에서는 다음 셀에서 `create_demo()`로 Gradio Blocks 객체를 확인할 수 있다.

In [ ]:
from week06 import create_demo

demo = create_demo()
demo

## 7. 회고

6주 동안 function call, structured output, RAG, MCP 형식 tool call, sub-agent, WebUI를 실제 API 호출 흐름으로 연결했다.
